In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

# index_col=0 supprime la colonne "Unnamed: 0", qui n'est qu'un numéro de ligne
train = pd.read_csv("train.csv", index_col=0)
test = pd.read_csv("test.csv", index_col=0)

print("train :", train.shape, "| test :", test.shape)
display(train.sample(3))
print(np.sort(train["year"].unique()))
display(train.dtypes)

train : (91927, 14) | test : (7609, 13)


,model_id,year,family_label,sub_department_label,department_label,univers_label,product_nature_label,brand_label,product_technicity_level,color_name,average_price,nb_store_max,nb_store_min,sales_quantity
47058,8390092,2021,BOY'S SWIMSUITS,SWIMMING,SWIMMING,WATER SPORTS UN,SWIMSUIT,ARENA,NaN,red,6.245,29,27,2128
4640,8551105,2020,WM BEG SSUIT,"SURF SWIMSUIT,",SURF BODYBOARD,WATER SPORTS UN,BIKINI TOP,OLAIAN,100,pink,9.450,808,626,2158
69491,8548307,2020,W MODERN CLOTH,DANCE,DANCE / GYMNAST,UNIVERS FITNESS,SPORTS BRA,STAREVER,500,smoked black,8.935,647,627,25425


[2019 2020 2021 2022]


model_id                      int64
year                          int64
family_label                    str
sub_department_label            str
department_label                str
univers_label                   str
product_nature_label            str
brand_label                     str
product_technicity_level        str
color_name                      str
average_price               float64
nb_store_max                  int64
nb_store_min                  int64
sales_quantity                int64
dtype: object

In [1]:
for col in ["color_name", "product_technicity_level"]:
    train[col] = train[col].fillna("INCONNU")

def evaluer(modele, colonnes_num, colonnes_cat=[]):
    # transforme les colonnes texte en colonnes 0/1 (one-hot)
    X = pd.get_dummies(train[colonnes_num + colonnes_cat], columns=colonnes_cat, dtype=float)
    y = train["sales_quantity"]
    entr = train["year"] < 2022    
      # on apprend sur 2019, 2020, 2021
    val = train["year"] == 2022          # on évalue sur 2022
    modele.fit(X[entr], y[entr])
    pred = np.clip(modele.predict(X[val]), 0, None)   # pas de ventes négatives
    rmse = np.sqrt(mean_squared_error(y[val], pred))
    print(f"{type(modele).__name__:<18} | {', '.join(colonnes_num + colonnes_cat):<60} | RMSE = {rmse:.0f}")
    return rmse

# Du plus simple au moins simple
resultats = {}

resultats["Dummy moyenne"] = evaluer(DummyRegressor(strategy="mean"), ["average_price"])
resultats["Lin nb_store_max"] = evaluer(LinearRegression(), ["nb_store_max"])
resultats["Lin 3 num"] = evaluer(LinearRegression(), ["average_price", "nb_store_max", "nb_store_min"])
resultats["Lin + univers"] = evaluer(LinearRegression(), ["average_price", "nb_store_max", "nb_store_min"], ["univers_label"])
#resultats["Lin + univers + niveau"] = evaluer(LinearRegression(), ["average_price", "nb_store_max", "nb_store_min"], ["univers_label", "product_technicity_level","family_label"])
resultats['DTR'] = evaluer(DecisionTreeRegressor(max_depth=3,random_state=0))
meilleur = min(resultats, key=resultats.get)
print("Meilleure méthode :", meilleur, "| RMSE =", f"{resultats[meilleur]:.0f}")

NameError: name 'train' is not defined